# T2.1 Phase 1 — L1 (5x) UNI Feature Extraction for the 9,128-slide PANDA cohort

**Purpose:** extract UNI ViT-L/16 features at 5x magnification (L1, cytology scale)
for the 8,528 slides not yet covered by `L1_P2_provider/uni/`. The existing 600
slides are detected via the progress file and skipped automatically.

**Q1 paper role:** unblocks `T2_1_phase3_v19_multiscale_run.ipynb` (the v19 L0+L1
multi-scale evaluation) which is the single biggest source of new content vs SIU/ICADA.

**Expected runtime on Colab Pro+ A100:** ~30-40 hours (chunked via partial-resume).
**Expected cost:** $0 (within Colab Pro+ $49.99/month subscription).

**Output schema** (matches existing 600 .pt files at L1_P2_provider/uni/):
```
{'features': torch.Tensor[N_patches, 1024],   # UNI 1024-d per patch
 'coordinates': torch.Tensor[N_patches, 2],   # (x, y) patch coords
 'config': 'P2_provider',                     # recipe tag
 'encoder': 'uni'}
```

**Parameters identical to L0 extraction** EXCEPT:
- `target_magnification`: 20 (L0) → **5 (L1)**
- `output_dir`: `UNI_features/` → **`CLAM_TR_Features/L1_P2_provider/uni/`**
- `progress_dir`: separate L1 progress file (does not interfere with L0)
- `.pt` save tag: `'L0_v20_full'` → **`'P2_provider'`**


## 1. Setup & Imports

In [1]:
# ── Install dependencies ──
import subprocess
for pkg in ['openslide-tools']:
    subprocess.run(['apt-get', 'install', '-q', pkg], capture_output=True)
for pkg in ['openslide-python', 'timm', 'huggingface_hub']:
    subprocess.run(['pip', 'install', '-q', pkg], capture_output=True)

# ── Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

# ── Imports ──
import os
import json
import time
import threading
import queue
import shutil
import numpy as np
from datetime import datetime
from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms
from tqdm import tqdm
import timm

import cv2
import openslide

from huggingface_hub import hf_hub_download, login as hf_login

# ── GPU ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── HuggingFace Auth (needed for MahmoodLab/UNI gated model) ──
# Get your token from: https://huggingface.co/settings/tokens
# Also request access at: https://huggingface.co/MahmoodLab/UNI
HF_TOKEN = "hf_YOUR_HF_TOKEN_HERE"  # <-- Replace with your actual token
hf_login(token=HF_TOKEN)
print("HuggingFace: Logged in")

print("Setup complete.")

Mounted at /content/drive
Device: cpu
HuggingFace: Logged in
Setup complete.


## 2. Configuration

In [2]:
# ══════════════════════════════════════════════════
# EDIT THESE PATHS FOR YOUR SETUP
# ══════════════════════════════════════════════════

# Where raw PANDA .tiff slides are on Google Drive
SLIDES_DIR = '/content/drive/MyDrive/PANDA_Subset_9128/images/'

# Where to save extracted .pt features (on Google Drive for persistence)
OUTPUT_DIR = '/content/drive/MyDrive/CLAM_TR_Features/L1_P2_provider/uni/'

# Where to save extraction progress (on Google Drive for persistence)
PROGRESS_DIR = '/content/drive/MyDrive/CLAM_TR_Preprocessing/L1_P2_provider/progress'

# ══════════════════════════════════════════════════
# PIPELINE SETTINGS (tune based on your resources)
# ══════════════════════════════════════════════════

EXTRACTION_CONFIG = {
    'slides_dir': SLIDES_DIR,
    'slide_ext': '.tiff',
    'output_dir': OUTPUT_DIR,
    'progress_dir': PROGRESS_DIR,
    'patch_size': 256,               # 256x256 patches (output size)
    'target_magnification': 5,    # L1 5x cytology scale (was 20 for L0)      # Extract at 20x (40x slides get resized)
    'encoder_batch_size': 256,       # Patches per GPU batch
    'checkpoint_every': 50,          # Save progress every N slides
    'min_tissue_ratio': 0.5,         # Min tissue percentage in patch
    'sat_thresh': 8,                 # HSV saturation threshold for tissue (CLAM default)
    'median_blur': 7,                # Median blur kernel for tissue seg (CLAM default)
    'morph_close': 4,                # Morphological closing kernel (CLAM default)
    'white_sat_thresh': 5,           # Max mean saturation for white patch (CLAM default)
    'black_rgb_thresh': 40,          # Max RGB mean for black patch (CLAM default)
    'min_patches_per_slide': 10,     # Skip slides with fewer patches
    'n_workers': 5,                  # CPU workers (copy + patch + transform)
    'retry_failed': True,              # Re-attempt previously failed slides
    'local_temp_dir': '/content/local_slides_temp',  # Temp dir for slides
}

# Create directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PROGRESS_DIR, exist_ok=True)

# Check slides
print("Configuration:")
for k, v in EXTRACTION_CONFIG.items():
    print(f"  {k}: {v}")

if os.path.isdir(SLIDES_DIR):
    slide_files = [f for f in os.listdir(SLIDES_DIR) if f.endswith('.tiff')]
    print(f"\nFound {len(slide_files)} slides in {SLIDES_DIR}")

    # Check existing features
    existing = len([f for f in os.listdir(OUTPUT_DIR) if f.endswith('.pt')]) if os.path.isdir(OUTPUT_DIR) else 0
    print(f"Existing features: {existing}")
    print(f"Remaining: ~{len(slide_files) - existing}")
else:
    print(f"\nWARNING: Slides not found at {SLIDES_DIR}")
    print("Update SLIDES_DIR above!")

Configuration:
  slides_dir: /content/drive/MyDrive/PANDA_Subset_9128/images/
  slide_ext: .tiff
  output_dir: /content/drive/MyDrive/UNI_features/
  progress_dir: /content/drive/MyDrive/CLAM_TR_Preprocessing/v20/progress
  patch_size: 256
  target_magnification: 20
  encoder_batch_size: 256
  checkpoint_every: 50
  min_tissue_ratio: 0.5
  sat_thresh: 8
  median_blur: 7
  morph_close: 4
  white_sat_thresh: 5
  black_rgb_thresh: 40
  min_patches_per_slide: 10
  n_workers: 5
  retry_failed: True
  local_temp_dir: /content/local_slides_temp

Found 9128 slides in /content/drive/MyDrive/PANDA_Subset_9128/images/
Existing features: 9128
Remaining: ~0


## 3. Extraction Pipeline

**Correctness features:**
- HSV saturation tissue segmentation (CLAM defaults: sat>8, blur=7, close=4)
- Magnification normalization (40x → read 512x512 → resize to 256x256)
- White patch filter: mean HSV saturation < 5 → skip (CLAM `isWhitePatch`)
- Black patch filter: all RGB means < 40 → skip (CLAM `isBlackPatch`)
- ImageNet normalization [0.485, 0.456, 0.406] / [0.229, 0.224, 0.225]
- Resize 256→224 for UNI ViT-L/16 input

```
N CPU workers (parallel)              1 GPU worker (main thread)
+-------------------------+          +--------------------------+
| Copy from Drive         |          |                          |
| HSV tissue segmentation |--+       |  Pull from GPU queue     |
| Magnification normalize |--+--> GPU Queue --> UNI inference   |
| White/black patch filter|--+   (max N)  |  Save .pt to Drive |
| Resize(224) + normalize |--+       |  Delete local slide      |
+-------------------------+          +--------------------------+
```

In [3]:
# ══════════════════════════════════════════════════════════════════
# Slide Preprocessor — CLAM-compatible tissue detection + magnification normalization
# ══════════════════════════════════════════════════════════════════

class SlidePreprocessor:
    """Extract tissue patches from WSI with magnification normalization and CLAM-style tissue detection.

    Correctness features vs naive approach:
    1. HSV saturation-based tissue segmentation (not Otsu on grayscale)
    2. Magnification normalization: 40x patches resized to match 20x physical area
    3. White patch filtering by HSV saturation (CLAM isWhitePatch)
    4. Black patch filtering by RGB threshold (CLAM isBlackPatch)
    """

    def __init__(self, patch_size=256, target_mag=20, min_tissue_ratio=0.5,
                 sat_thresh=8, median_blur=7, morph_close=4,
                 white_sat_thresh=5, black_rgb_thresh=40):
        self.patch_size = patch_size
        self.target_mag = target_mag
        self.min_tissue_ratio = min_tissue_ratio
        self.sat_thresh = sat_thresh
        self.median_blur = median_blur
        self.morph_close = morph_close
        self.white_sat_thresh = white_sat_thresh
        self.black_rgb_thresh = black_rgb_thresh

    def get_extraction_params(self, slide):
        """Find best pyramid level and calculate read/step sizes for target magnification.

        For 40x slides without a native 20x level:
          reads 512x512 at 40x level 0 and resizes to 256x256,
          ensuring patches cover the same physical tissue area as 256x256 at 20x.
        For 20x slides: reads 256x256 directly (no resize needed).

        Returns: (level, read_size, step_l0, actual_mag, obj_power)
        """
        try:
            obj_power = float(slide.properties.get('openslide.objective-power', 20))
        except (ValueError, KeyError):
            obj_power = 20

        downsample_needed = obj_power / self.target_mag
        best_level, best_diff = 0, float('inf')
        for level in range(slide.level_count):
            diff = abs(slide.level_downsamples[level] - downsample_needed)
            if diff < best_diff:
                best_diff, best_level = diff, level

        level_ds = slide.level_downsamples[best_level]
        actual_mag = obj_power / level_ds
        mag_ratio = actual_mag / self.target_mag  # >1 means higher mag than target

        # Read larger region if mag is higher, then resize to patch_size
        read_size = int(round(self.patch_size * mag_ratio))
        # Step in level 0 coordinates (non-overlapping physical coverage)
        step_l0 = int(round(read_size * level_ds))

        return best_level, read_size, step_l0, actual_mag, obj_power

    def segment_tissue(self, slide):
        """HSV saturation-based tissue segmentation (matches CLAM pipeline).

        Steps: RGB -> HSV -> Saturation channel -> Median blur -> Binary threshold -> Morphological closing
        CLAM defaults: sat_thresh=8, median_blur=7, morph_close=4
        """
        # Auto-select low-resolution level for segmentation (~64x downsample)
        seg_level = 0
        for level in range(slide.level_count):
            if slide.level_downsamples[level] <= 64:
                seg_level = level

        dims = slide.level_dimensions[seg_level]
        img = np.array(slide.read_region((0, 0), seg_level, dims).convert('RGB'))

        # HSV -> Saturation -> Median blur -> Binary threshold
        img_hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
        img_med = cv2.medianBlur(img_hsv[:, :, 1], self.median_blur)
        _, tissue_mask = cv2.threshold(img_med, self.sat_thresh, 255, cv2.THRESH_BINARY)

        # Morphological closing to fill small gaps in tissue
        if self.morph_close > 0:
            kernel = np.ones((self.morph_close, self.morph_close), np.uint8)
            tissue_mask = cv2.morphologyEx(tissue_mask, cv2.MORPH_CLOSE, kernel)

        return tissue_mask, seg_level

    def extract_patches_and_coords(self, slide):
        """Extract tissue patches at target magnification with proper normalization.

        Key behavior:
        - 20x slides: reads 256x256 patches directly
        - 40x slides (no 20x level): reads 512x512, resizes to 256x256
          -> same physical area coverage, consistent feature representations
        - Filters: HSV tissue mask + white patch (sat<5) + black patch (rgb<40)
        """
        level, read_size, step_l0, actual_mag, obj_power = self.get_extraction_params(slide)
        tissue_mask, seg_level = self.segment_tissue(slide)
        seg_ds = slide.level_downsamples[seg_level]
        mask_h, mask_w = tissue_mask.shape
        needs_resize = (read_size != self.patch_size)

        patches, coords = [], []
        w0, h0 = slide.level_dimensions[0]

        for y in range(0, h0 - step_l0 + 1, step_l0):
            for x in range(0, w0 - step_l0 + 1, step_l0):
                # Check tissue ratio on segmentation mask
                mx = min(int(x / seg_ds), mask_w - 1)
                my = min(int(y / seg_ds), mask_h - 1)
                mw = max(1, min(int(step_l0 / seg_ds), mask_w - mx))
                mh = max(1, min(int(step_l0 / seg_ds), mask_h - my))
                if mw <= 0 or mh <= 0:
                    continue

                tissue_ratio = np.mean(tissue_mask[my:my+mh, mx:mx+mw] > 0)
                if tissue_ratio < self.min_tissue_ratio:
                    continue

                try:
                    patch_pil = slide.read_region((x, y), level,
                                                  (read_size, read_size)).convert('RGB')
                    patch_arr = np.array(patch_pil)

                    # White patch check: low saturation = background (CLAM isWhitePatch)
                    patch_hsv = cv2.cvtColor(patch_arr, cv2.COLOR_RGB2HSV)
                    if np.mean(patch_hsv[:, :, 1]) < self.white_sat_thresh:
                        continue

                    # Black patch check: all channels dark = artifact (CLAM isBlackPatch)
                    if np.all(np.mean(patch_arr, axis=(0, 1)) < self.black_rgb_thresh):
                        continue

                    # Magnification normalization: resize if higher mag than target
                    if needs_resize:
                        patch_pil = patch_pil.resize((self.patch_size, self.patch_size),
                                                     Image.LANCZOS)
                        patch_arr = np.array(patch_pil)

                    patches.append(patch_arr)
                    coords.append([x, y])
                except Exception:
                    continue

        return patches, np.array(coords, dtype=np.int32) if coords else np.zeros((0, 2), dtype=np.int32)


# ══════════════════════════════════════════════════════════════════
# UNI v1 Encoder
# ══════════════════════════════════════════════════════════════════

class UNIv1Encoder(nn.Module):
    """UNI v1 (ViT-L/16, 1024-dim) from MahmoodLab."""

    def __init__(self):
        super().__init__()
        print("Loading UNI v1 (ViT-L/16)...")
        self.encoder = timm.create_model(
            "vit_large_patch16_224",
            img_size=224, patch_size=16,
            init_values=1e-5,
            num_classes=0,
            dynamic_img_size=True
        )
        weights_path = hf_hub_download("MahmoodLab/UNI", "pytorch_model.bin")
        self.encoder.load_state_dict(
            torch.load(weights_path, map_location="cpu", weights_only=False), strict=True)
        for p in self.encoder.parameters():
            p.requires_grad = False
        print(f"UNI v1 loaded: {sum(p.numel() for p in self.parameters()):,} params (frozen)")

    def forward(self, x):
        return self.encoder(x)


# ══════════════════════════════════════════════════════════════════
# Slide Reading Verification
# ══════════════════════════════════════════════════════════════════

def verify_slide_reading(slides_dir, slide_ext, n_samples=5):
    """Verify OpenSlide reads slides correctly before full extraction."""
    print("=" * 60)
    print("SLIDE READING VERIFICATION")
    print("=" * 60)

    slide_files = sorted([f for f in os.listdir(slides_dir) if f.endswith(slide_ext)])
    if not slide_files:
        print(f"  ERROR: No {slide_ext} files in {slides_dir}")
        return False

    import random
    random.seed(42)
    samples = random.sample(slide_files, min(n_samples, len(slide_files)))
    issues = []
    test_preprocessor = SlidePreprocessor()

    for fname in samples:
        fpath = os.path.join(slides_dir, fname)
        slide = openslide.OpenSlide(fpath)
        level, read_size, step_l0, actual_mag, obj_power = test_preprocessor.get_extraction_params(slide)
        level_dims = slide.level_dimensions[level]
        needs_resize = (read_size != 256)

        # Higher mag is fine (patches get resized to correct physical area)
        # Only flag if effective mag is LOWER than target
        if actual_mag >= 19.0:
            status = "OK"
        elif actual_mag >= 10.0:
            status = "WARNING"
            issues.append(f"{fname}: low mag {actual_mag:.1f}x (target 20x)")
        else:
            status = "FAIL"
            issues.append(f"{fname}: very low mag {actual_mag:.1f}x")

        resize_note = f" -> resize {read_size}->256" if needs_resize else ""
        print(f"  {fname}: obj={obj_power}x, level={level} "
              f"({level_dims[0]}x{level_dims[1]}), eff={actual_mag:.1f}x, "
              f"read={read_size}x{read_size}{resize_note} [{status}]")

        try:
            test_patch = slide.read_region((0, 0), level, (read_size, read_size)).convert('RGB')
            arr = np.array(test_patch)
            hsv = cv2.cvtColor(arr, cv2.COLOR_RGB2HSV)
            print(f"    Patch: {arr.shape}, mean_rgb={arr.mean():.1f}, mean_sat={hsv[:,:,1].mean():.1f}")
        except Exception as e:
            print(f"    Test patch FAILED: {e}")
            issues.append(f"{fname}: read_region failed")

        slide.close()

    if issues:
        print(f"\n  ISSUES: {len(issues)}")
        for i in issues:
            print(f"    - {i}")
        return False
    print(f"\n  All {len(samples)} slides OK")
    return True


# ══════════════════════════════════════════════════════════════════
# Multi-Worker Feature Extraction Pipeline
# ══════════════════════════════════════════════════════════════════

def extract_uni_features():
    """Extract UNI v1 features with multi-worker pipeline.

    N CPU workers (copy + patch + transform) feed a GPU queue.
    1 GPU worker (main thread) runs UNI inference and saves .pt files.
    Local slides are deleted after extraction to save disk space.
    """
    cfg = EXTRACTION_CONFIG
    N_WORKERS = cfg['n_workers']
    LOCAL_TEMP = cfg['local_temp_dir']

    # Progress tracking (on Drive for crash persistence)
    progress_path = os.path.join(cfg['progress_dir'], 'uni_extraction_progress.json')
    if os.path.exists(progress_path):
        with open(progress_path, 'r') as f:
            progress = json.load(f)
        completed = set(progress.get('completed', []))
        failed_prev = set(progress.get('failed', []))
        insufficient_tissue_prev = set(progress.get('insufficient_tissue', []))
    else:
        completed, failed_prev, insufficient_tissue_prev = set(), set(), set()

    # Get all slide IDs
    slide_files = sorted([f for f in os.listdir(cfg['slides_dir'])
                          if f.endswith(cfg['slide_ext'])])
    all_ids = [f.replace(cfg['slide_ext'], '') for f in slide_files]

    # Check existing .pt features
    existing_pts = set()
    if os.path.isdir(cfg['output_dir']):
        existing_pts = {f.replace('.pt', '') for f in os.listdir(cfg['output_dir'])
                        if f.endswith('.pt')}
    completed = completed | existing_pts

    retry_failed = cfg.get('retry_failed', True)
    # insufficient_tissue_prev is a permanent-skip list: slides whose tissue floor
    # is below cfg['min_patches_per_slide'] at this preprocessing config never recover
    # on retry, so they are excluded regardless of retry_failed.
    if retry_failed:
        to_process = [sid for sid in all_ids if sid not in completed and sid not in insufficient_tissue_prev]
    else:
        to_process = [sid for sid in all_ids if sid not in completed and sid not in failed_prev and sid not in insufficient_tissue_prev]

    print("=" * 60)
    print("UNI v1 FEATURE EXTRACTION (Multi-Worker Pipeline)")
    print("=" * 60)
    print(f"Total slides:      {len(all_ids)}")
    print(f"Already done:      {len(completed)}")
    print(f"Previously failed: {len(failed_prev)}")
    print(f"Permanently skipped (insufficient tissue): {len(insufficient_tissue_prev)}")
    print(f"To process:        {len(to_process)}")
    print(f"Output dir:        {cfg['output_dir']}")
    print(f"CPU workers:       {N_WORKERS}")
    print(f"Tissue seg:        HSV saturation (sat>{cfg.get('sat_thresh',8)}, blur={cfg.get('median_blur',7)})")
    print(f"Patch filtering:   white(sat<{cfg.get('white_sat_thresh',5)}) + black(rgb<{cfg.get('black_rgb_thresh',40)})")
    print(f"Magnification:     target {cfg['target_magnification']}x (40x slides auto-resized)")

    if not to_process:
        print("\nAll slides already processed!")
        return

    # Verify slides first
    print()
    if not verify_slide_reading(cfg['slides_dir'], cfg['slide_ext']):
        resp = input("Issues detected. Continue? (y/n): ").strip().lower()
        if resp != 'y':
            return
    print()

    # Load UNI model (ONE model, shared across threads for inference)
    encoder = UNIv1Encoder().to(device).eval()

    # ImageNet normalization constants (GPU batch transform replaces CPU img_transform)
    _gpu_mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    _gpu_std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

    preprocessor = SlidePreprocessor(
        patch_size=cfg['patch_size'],
        target_mag=cfg['target_magnification'],
        min_tissue_ratio=cfg['min_tissue_ratio'],
        sat_thresh=cfg.get('sat_thresh', 8),
        median_blur=cfg.get('median_blur', 7),
        morph_close=cfg.get('morph_close', 4),
        white_sat_thresh=cfg.get('white_sat_thresh', 5),
        black_rgb_thresh=cfg.get('black_rgb_thresh', 40),
    )

    # ── Queues ──
    os.makedirs(LOCAL_TEMP, exist_ok=True)
    slide_ids_q = queue.Queue()
    for sid in to_process:
        slide_ids_q.put(sid)

    gpu_q = queue.Queue(maxsize=N_WORKERS)
    preprocess_errors = []
    # In-session permanent-skip set: seeded from prev, grows when a new slide
    # is found to have insufficient tissue at this preprocessing config.
    insufficient_tissue = set(insufficient_tissue_prev)
    _lock = threading.Lock()

    def _cpu_worker():
        """Read slide (Drive direct or local fallback), extract patches as raw numpy."""
        while True:
            try:
                slide_id = slide_ids_q.get_nowait()
            except Exception:
                return

            src_path = os.path.join(cfg['slides_dir'], f"{slide_id}{cfg['slide_ext']}")
            local_path = os.path.join(LOCAL_TEMP, f"{slide_id}{cfg['slide_ext']}")
            used_local = False

            try:
                # Try reading directly from Drive (skip copy = ~15 sec saved per slide)
                try:
                    slide = openslide.OpenSlide(src_path)
                except Exception:
                    # Fallback: copy to local then open
                    shutil.copy2(src_path, local_path)
                    used_local = True
                    slide = openslide.OpenSlide(local_path)

                patches, coords = preprocessor.extract_patches_and_coords(slide)
                slide.close()

                if len(patches) < cfg['min_patches_per_slide']:
                    # Tissue-floor reject: slide has insufficient patches at the chosen
                    # preprocessing config. Track in insufficient_tissue (permanent skip);
                    # NOT a preprocess error and NOT eligible for retry.
                    with _lock:
                        insufficient_tissue.add(slide_id)
                    gpu_q.put((slide_id, None, None, local_path if used_local else None))
                    continue

                patch_tensors = torch.from_numpy(np.stack(patches))  # (N, 256, 256, 3) uint8
                coords_tensor = torch.tensor(coords, dtype=torch.float32)
                del patches

                gpu_q.put((slide_id, patch_tensors, coords_tensor, local_path if used_local else None))

            except Exception as e:
                with _lock:
                    preprocess_errors.append((slide_id, str(e)))
                gpu_q.put((slide_id, None, None, local_path if used_local and os.path.exists(local_path) else None))

    # ── Start CPU workers ──
    actual_workers = min(N_WORKERS, len(to_process))
    cpu_threads = []
    for _ in range(actual_workers):
        t = threading.Thread(target=_cpu_worker, daemon=True)
        t.start()
        cpu_threads.append(t)
    print(f"  {actual_workers} CPU workers started")

    # ── GPU loop (main thread) ──
    completed_list = list(completed)
    failed_list = list(failed_prev)
    batch_size = cfg['encoder_batch_size']
    start_time = time.time()
    session_count = 0
    processed_count = 0
    total = len(to_process)

    pbar = tqdm(total=total, desc="Extracting", unit="slide")

    while processed_count < total:
        try:
            item = gpu_q.get(timeout=300)
        except Exception:
            alive = sum(1 for t in cpu_threads if t.is_alive())
            print(f"\n  Queue timeout. {alive}/{actual_workers} workers alive")
            if alive == 0:
                break
            continue

        slide_id, patch_tensors, coords_tensor, local_path = item
        processed_count += 1

        if patch_tensors is None:
            # Distinguish tissue-floor reject (insufficient_tissue, permanent skip)
            # from genuine slide-read error (failed_list, eligible for retry).
            if slide_id not in insufficient_tissue:
                failed_list.append(slide_id)
            pbar.update(1)
            try:
                if local_path and os.path.exists(local_path):
                    os.remove(local_path)
            except Exception:
                pass
            continue

        try:
            # GPU transform + inference (fp16 autocast, fp32 output)
            all_features = []
            with torch.inference_mode(), torch.amp.autocast('cuda'):
                for b in range(0, len(patch_tensors), batch_size):
                    batch_raw = patch_tensors[b:b+batch_size]  # (B, 256, 256, 3) uint8
                    batch_gpu = batch_raw.to(device).permute(0, 3, 1, 2).float() / 255.0  # (B, 3, 256, 256)
                    batch_gpu = torch.nn.functional.interpolate(batch_gpu, size=(224, 224), mode='bilinear', align_corners=False)
                    batch_gpu = (batch_gpu - _gpu_mean) / _gpu_std
                    feats = encoder(batch_gpu).float().cpu()  # .float() ensures fp32 output
                    all_features.append(feats)
                    del batch_raw, batch_gpu

            features_cat = torch.cat(all_features, dim=0)

            # Save .pt (format: {'features': [N, 1024], 'coords': [N, 2]})
            output_path = os.path.join(cfg['output_dir'], f"{slide_id}.pt")
            torch.save({'features': features_cat, 'coords': coords_tensor}, output_path)

            completed_list.append(slide_id)
            session_count += 1
            del patch_tensors, all_features, features_cat, coords_tensor

            elapsed = time.time() - start_time
            rate = session_count / (elapsed / 60)
            remaining = (total - processed_count) / max(rate, 0.01)
            pbar.set_postfix(rate=f"{rate:.1f}/min", eta=f"{remaining:.0f}min")

        except Exception as e:
            print(f"\n  GPU ERROR {slide_id}: {e}")
            failed_list.append(slide_id)

        finally:
            try:
                if local_path and os.path.exists(local_path):
                    os.remove(local_path)
            except Exception:
                pass

        pbar.update(1)

        # Checkpoint
        if session_count % cfg['checkpoint_every'] == 0 and session_count > 0:
            with open(progress_path, 'w') as f:
                json.dump({
                    'completed': completed_list,
                    'failed': failed_list,
                    'insufficient_tissue': sorted(insufficient_tissue),
                    'total': len(all_ids),
                    'timestamp': datetime.now().isoformat()
                }, f)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    pbar.close()

    for t in cpu_threads:
        t.join(timeout=10)

    # Final save
    with open(progress_path, 'w') as f:
        json.dump({
            'completed': completed_list,
            'failed': failed_list,
            'insufficient_tissue': sorted(insufficient_tissue),
            'total': len(all_ids),
            'timestamp': datetime.now().isoformat()
        }, f)

    elapsed = time.time() - start_time
    print(f"\n{'=' * 60}")
    print(f"EXTRACTION COMPLETE")
    print(f"  This session: {session_count} slides in {elapsed/60:.1f} min")
    print(f"  Total: {len(completed_list)}/{len(all_ids)} done, {len(failed_list)} failed")
    new_insufficient = len(insufficient_tissue) - len(insufficient_tissue_prev)
    print(f"  Insufficient tissue (permanent-skip): {len(insufficient_tissue)} total (+{new_insufficient} this session)")
    if preprocess_errors:
        print(f"  Genuine preprocess errors (slide-read/processing): {len(preprocess_errors)}")
        for sid, err in preprocess_errors[:5]:
            print(f"    - {sid}: {err}")
    print(f"  Output: {cfg['output_dir']}")
    print(f"{'=' * 60}")

    # Cleanup
    shutil.rmtree(LOCAL_TEMP, ignore_errors=True)
    del encoder
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Pipeline ready.")
print("  SlidePreprocessor: HSV tissue seg + mag normalization + 256x256 patches")
print("  UNIv1Encoder: ViT-L/16, 1024-dim (MahmoodLab/UNI)")
print("  extract_uni_features(): multi-worker CPU + 1 GPU pipeline")

Pipeline ready.
  SlidePreprocessor: HSV tissue seg + mag normalization + 256x256 patches
  UNIv1Encoder: ViT-L/16, 1024-dim (MahmoodLab/UNI)
  extract_uni_features(): multi-worker CPU + 1 GPU pipeline


## 4. Run Extraction

In [4]:
# ── Run UNI v1 Feature Extraction ──
# This will:
#   1. Check existing features (skip already-extracted slides)
#   2. Verify slide reading on 5 sample slides
#   3. Start CPU workers + 1 GPU worker pipeline
#   4. Save .pt features to Google Drive
#   5. Checkpoint every 50 slides (safe to restart)

extract_uni_features()

UNI v1 FEATURE EXTRACTION (Multi-Worker Pipeline)
Total slides:      9128
Already done:      9128
Previously failed: 0
To process:        0
Output dir:        /content/drive/MyDrive/UNI_features/
CPU workers:       5
Tissue seg:        HSV saturation (sat>8, blur=7)
Patch filtering:   white(sat<5) + black(rgb<40)
Magnification:     target 20x (40x slides auto-resized)

All slides already processed!


## 5. Verify Extracted Features

In [5]:
# Check extraction results
output_dir = EXTRACTION_CONFIG['output_dir']
pt_files = sorted([f for f in os.listdir(output_dir) if f.endswith('.pt')])
print(f"Total .pt features: {len(pt_files)}")

# Check a few samples
import random
random.seed(42)
samples = random.sample(pt_files, min(5, len(pt_files)))
print(f"\nSample features:")
for fname in samples:
    fpath = os.path.join(output_dir, fname)
    data = torch.load(fpath, map_location='cpu', weights_only=False)
    feat = data['features']
    coords = data['coords']
    print(f"  {fname}: features={feat.shape}, coords={coords.shape}, "
          f"mean={feat.mean():.4f}, std={feat.std():.4f}")

# Check progress file
progress_path = os.path.join(EXTRACTION_CONFIG['progress_dir'], 'uni_extraction_progress.json')
if os.path.exists(progress_path):
    with open(progress_path) as f:
        prog = json.load(f)
    print(f"\nProgress file:")
    print(f"  Completed: {len(prog.get('completed', []))}")
    print(f"  Failed: {len(prog.get('failed', []))}")
    print(f"  Total: {prog.get('total', '?')}")
    print(f"  Last saved: {prog.get('timestamp', '?')}")

Total .pt features: 9128

Sample features:
  34b8869c4b4ba4fdfea0a9e579502c5f.pt: features=torch.Size([403, 1024]), coords=torch.Size([403, 2]), mean=-0.0067, std=1.1947
  0bf60b57d6fbd60feb3923848bdc720a.pt: features=torch.Size([285, 1024]), coords=torch.Size([285, 2]), mean=-0.0091, std=1.1943
  81c391aa0e95da9f8cb3419bce40236c.pt: features=torch.Size([503, 1024]), coords=torch.Size([503, 2]), mean=-0.0150, std=1.2280
  73df680f7884e13355a1f1fc2bacc3a4.pt: features=torch.Size([466, 1024]), coords=torch.Size([466, 2]), mean=-0.0112, std=1.2054
  69ed9b36a92904dcee7b78c338e5f4f4.pt: features=torch.Size([569, 1024]), coords=torch.Size([569, 2]), mean=-0.0150, std=1.2092

Progress file:
  Completed: 9128
  Failed: 0
  Total: 9128
  Last saved: 2026-03-01T06:48:38.271349


## 6. Create tar.gz Archive

Creates `uni_features.tar.gz` on Drive for fast local copy in training notebooks.
One large file copy avoids Drive FUSE freeze on thousands of small files.

In [ ]:
# ── Create tar.gz of extracted features ──
# Drive FUSE freezes when reading thousands of small files.
# Solution: batch-copy .pt files to local SSD, tar locally,
# then copy single tar back to Drive.

import tarfile, subprocess

output_dir = EXTRACTION_CONFIG['output_dir']
tar_drive = os.path.join(output_dir, 'uni_features.tar.gz')

if os.path.exists(tar_drive):
    print(f'tar.gz already exists: {tar_drive}')
    print(f'Size: {os.path.getsize(tar_drive)/1e9:.1f} GB')
    print('Delete it first if you want to recreate.')
else:
    local_pt_dir = '/content/local_pt_temp'
    tar_local = '/content/uni_features.tar.gz'
    os.makedirs(local_pt_dir, exist_ok=True)

    pt_files = sorted([f for f in os.listdir(output_dir) if f.endswith('.pt')])
    print(f'Features: {len(pt_files)} .pt files')

    # Step 1: Batch-copy .pt files from Drive to local
    # Use shell cp in batches of 50 with sync to prevent FUSE freeze
    print(f'Step 1: Copying .pt files to local disk...')
    batch_size = 50
    pbar = tqdm(total=len(pt_files), desc='Copy to local', unit='file')
    for i in range(0, len(pt_files), batch_size):
        batch = pt_files[i:i+batch_size]
        for f in batch:
            src = os.path.join(output_dir, f)
            dst = os.path.join(local_pt_dir, f)
            if not os.path.exists(dst):
                subprocess.run(['cp', src, dst], check=True)
            pbar.update(1)
        # Sync after each batch to keep FUSE healthy
        subprocess.run(['sync'], check=True)
    pbar.close()

    local_count = len([f for f in os.listdir(local_pt_dir) if f.endswith('.pt')])
    print(f'Local files: {local_count}')

    # Step 2: Create tar.gz from local files (fast — no Drive I/O)
    print(f'Step 2: Creating tar.gz from local files...')
    local_pts = sorted([f for f in os.listdir(local_pt_dir) if f.endswith('.pt')])
    with tarfile.open(tar_local, 'w:gz') as tar:
        for f in tqdm(local_pts, desc='Archiving', unit='file'):
            tar.add(os.path.join(local_pt_dir, f), arcname=f)

    local_size = os.path.getsize(tar_local) / 1e9
    print(f'tar.gz size: {local_size:.1f} GB')

    # Step 3: Copy single tar to Drive
    print(f'Step 3: Uploading tar.gz to Drive...')
    total = os.path.getsize(tar_local)
    chunk_size = 64 * 1024 * 1024
    pbar = tqdm(total=total, unit='B', unit_scale=True, desc='Upload')
    with open(tar_local, 'rb') as src, open(tar_drive, 'wb') as dst:
        while True:
            chunk = src.read(chunk_size)
            if not chunk:
                break
            dst.write(chunk)
            pbar.update(len(chunk))
    pbar.close()

    # Cleanup
    import shutil
    shutil.rmtree(local_pt_dir)
    os.remove(tar_local)
    print(f'sDone: {tar_drive} ({local_size:.1f} GB, {len(pt_files)} files)')


Features: 9128 .pt files
Step 1: Copying .pt files to local disk...


Copy to local:   1%|          | 87/9128 [00:32<37:57,  3.97file/s]